In [0]:
# Configura o catálogo e cria o schema da camada Silver

catalog = "workspace"
silver_schema = "silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}")

print(f"Schema criado: {catalog}.{silver_schema}")


In [0]:
# Carrega a tabela de informações dos filmes para analisar os dados

df_info = spark.table("workspace.bronze.tb_movies_info")

display(df_info)


In [0]:
# Tratamento da tabela de informações dos filmes

from pyspark.sql import Window
from pyspark.sql.functions import (
    col, trim, lower, regexp_replace, coalesce, expr, when,
    row_number, desc_nulls_last, lit
)

df_info_silver = (
    df_info
    .withColumn("id", trim(col("id")))
    .withColumn("tconst", trim(col("tconst")))
    .withColumn("title", trim(col("title")))
    .withColumn("original_title", trim(col("original_title")))
    .withColumn("original_language", trim(col("original_language")))
    .withColumn("status", trim(col("status")))
    .withColumn("overview", trim(col("overview")))
    .withColumn("tagline", trim(col("tagline")))

    # Padroniza a data de lançamento
    .withColumn(
        "release_date",
        coalesce(
            expr("try_to_date(release_date, 'yyyy-MM-dd')"),
            expr("try_to_date(release_date, 'MM-dd-yyyy')"),
            expr("try_to_date(release_date, 'dd/MM/yyyy')"),
            expr("try_to_date(release_date, 'MM/dd/yyyy')")
        )
    )

    # Converte a duração para número inteiro
    .withColumn(
        "runtime",
        when(
            trim(col("runtime")).rlike(r"^\d+$"),
            trim(col("runtime")).cast("int")
        )
    )

    # Padroniza o status
    .withColumn(
        "status",
        when(
            regexp_replace(lower(col("status")), "-", " ") == "released",
            "Released"
        )
        .when(
            regexp_replace(lower(col("status")), "-", " ") == "in production",
            "In Production"
        )
        .when(
            regexp_replace(lower(col("status")), "-", " ") == "post production",
            "Post Production"
        )
        .when(lower(col("status")) == "planned", "Planned")
        .otherwise(None)
    )

    # Descarta registros sem identificador
    .filter(col("id").isNotNull() & (col("id") != ""))
)

# Conta os campos preenchidos para desempatar registros do mesmo filme
campos_informativos = [
    "tconst", "title", "original_title", "original_language",
    "release_date", "runtime", "status", "overview", "tagline"
]

pontuacao = lit(0)
for campo in campos_informativos:
    pontuacao = pontuacao + when(
        col(campo).isNotNull() & (trim(col(campo).cast("string")) != ""),
        1
    ).otherwise(0)

df_info_silver = df_info_silver.withColumn(
    "campos_preenchidos", pontuacao
)

# Seleciona um registro por ID, priorizando a ingestão mais recente

janela_filmes = Window.partitionBy("id").orderBy(
    desc_nulls_last("ingestion_datetime"),
    col("campos_preenchidos").desc(),
    col("title").asc_nulls_last(),
    col("tconst").asc_nulls_last(),
    col("original_title").asc_nulls_last(),
    col("release_date").asc_nulls_last(),
    col("runtime").asc_nulls_last(),
    col("original_language").asc_nulls_last(),
    col("status").asc_nulls_last(),
    col("overview").asc_nulls_last(),
    col("tagline").asc_nulls_last()
)

df_info_silver = (
    df_info_silver
    .withColumn("rn", row_number().over(janela_filmes))
    .filter(col("rn") == 1)
    .drop("rn", "campos_preenchidos")
)


In [0]:
# Carrega os dados financeiros da camada Bronze

df_financeiro = spark.table("workspace.bronze.tb_movies_financials")

print("Quantidade de registros:", df_financeiro.count())

display(df_financeiro.limit(20))


In [0]:
# Limpa budget e revenue e converte os valores financeiros para número

from pyspark.sql.functions import col, when, regexp_replace, upper, trim

def limpar_valor_financeiro(coluna):
    valor = upper(trim(col(coluna)))

    numero_limpo = regexp_replace(valor, r"[\$ USD,]", "")

    return (
        when(
            valor.isNull() |
            valor.isin("UNKNOWN", "NÃO INFORMADO", "NAO INFORMADO", "N/A", ""),
            None
        )
        # Valores em milhões

        .when(
            valor.rlike(r"^[\$ ]*(USD )?\d+(\.\d+)?M$"),
            regexp_replace(valor, r"[\$ USDM,]", "").cast("double") * 1000000
        )
        # Valores em milhares

        .when(
            valor.rlike(r"^[\$ ]*(USD )?\d+(\.\d+)?K$"),
            regexp_replace(valor, r"[\$ USDK,]", "").cast("double") * 1000
        )
        # Valores numéricos comuns

        .when(
            numero_limpo.rlike(r"^\d+(\.\d+)?$"),
            numero_limpo.cast("double")
        )
        # Qualquer outro formato inválido vira NULL
        
        .otherwise(None)
    )

df_financeiro_silver = (
    df_financeiro
    .withColumn("id", trim(col("id")))
    .withColumn("budget", limpar_valor_financeiro("budget"))
    .withColumn("revenue", limpar_valor_financeiro("revenue"))
    .dropDuplicates(["id", "budget", "revenue"])
)

display(df_financeiro_silver.limit(20))



In [0]:
# Carrega os dados de métricas da camada Bronze

df_metricas = spark.table("workspace.bronze.tb_movies_metrics")

print("Quantidade de registros:", df_metricas.count())

display(df_metricas.limit(20))


In [0]:
# Trata as métricas numéricas e avaliações dos filmes

from pyspark.sql.functions import col, trim, when

def converter_numero(coluna):
    valor = trim(col(coluna))

    return (
        when(
            valor.rlike(r"^-?\d+(\.\d+)?$"),
            valor.cast("double")
        )
        .otherwise(None)
    )

df_metricas_silver = (
    df_metricas
    .withColumn("id", trim(col("id")))
    .withColumn("popularity", converter_numero("popularity"))
    .withColumn("vote_average", converter_numero("vote_average"))
    .withColumn("vote_count", converter_numero("vote_count"))
    .withColumn("averageRating", converter_numero("averageRating"))
    .withColumn("numVotes", converter_numero("numVotes"))

    # Avaliações válidas devem ficar entre 0 e 10

    .withColumn(
        "vote_average",
        when(
            col("vote_average").between(0, 10),
            col("vote_average")
        ).otherwise(None)
    )
    .withColumn(
        "averageRating",
        when(
            col("averageRating").between(0, 10),
            col("averageRating")
        ).otherwise(None)
    )

    # Remove registros completamente duplicados

    .dropDuplicates()
)

display(df_metricas_silver.limit(20))



In [0]:
# Carrega os dados de avaliações da camada Bronze

df_avaliacoes = spark.table("workspace.bronze.tb_movies_reviews")

print("Quantidade de registros:", df_avaliacoes.count())

df_avaliacoes.printSchema()

display(df_avaliacoes.limit(20))


In [0]:
# Tratamento dos dados de avaliações para a camada Silver

from pyspark.sql.functions import col, trim, when, expr

df_avaliacoes_silver = (
    df_avaliacoes
    .withColumn("id", trim(col("id")))
    .withColumn("nome", trim(col("nome")))
    .withColumn(
        "comentario",
        when(
            col("comentario").isNull() | (trim(col("comentario")) == ""),
            None
        ).otherwise(trim(col("comentario")))
    )
    .withColumn(
        "nota",
        expr("try_cast(trim(nota) as double)")
    )
    .withColumn(
        "nota",
        when(col("nota").between(0, 10), col("nota"))
        .otherwise(None)
    )
)



In [0]:
# Carrega os dados de créditos e tags da camada Bronze

df_credits = spark.table("workspace.bronze.tb_credits_and_tags")

print("Quantidade de registros:", df_credits.count())

df_credits.printSchema()

display(df_credits.limit(20))


In [0]:
from pyspark.sql.functions import col, trim, regexp_replace, split, explode

df_generos_silver = (
    df_credits
    # Padroniza "|" para "," para termos um único separador
    
    .withColumn("genres", regexp_replace(col("genres"), r"\|", ","))

    # Separa os vários gêneros e transforma cada um em uma linha

    .withColumn("genero", explode(split(col("genres"), ",")))

    # Remove espaços extras

    .withColumn("genero", trim(col("genero")))

    # Mantém apenas as colunas necessárias

    .select(
        trim(col("id")).alias("id"),
        col("genero"),
        col("ingestion_datetime")
    )

    # Remove gêneros vazios ou nulos

    .filter(
        col("genero").isNotNull() &
        (col("genero") != "")
    )

    # Remove repetições do mesmo gênero para o mesmo filme

    .dropDuplicates(["id", "genero"])
)

display(df_generos_silver.limit(30))


In [0]:
# Lista de gêneros válidos encontrados na base
generos_validos = [
    "Action",
    "Adventure",
    "Animation",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Family",
    "Fantasy",
    "History",
    "Horror",
    "Music",
    "Mystery",
    "Romance",
    "Science Fiction",
    "TV Movie",
    "Thriller",
    "War",
    "Western"
]

# Mantém somente os valores que realmente representam gêneros

df_generos_silver = (
    df_generos_silver
    .filter(col("genero").isin(generos_validos))
)

# Confere o resultado final

print("Quantidade de registros:", df_generos_silver.count())
print(
    "Quantidade de gêneros diferentes:",
    df_generos_silver.select("genero").distinct().count()
)

df_generos_silver.select("genero").distinct().orderBy("genero").show(30, truncate=False)


In [0]:
# Carrega os dados de créditos e tags da camada Bronze para iniciar o tratamento de pessoas e empresas

df_pessoas = spark.table("workspace.bronze.tb_credits_and_tags")

df_pessoas.printSchema()

display(df_pessoas.limit(20))


In [0]:
# Extração e tratamento de pessoas e empresas

from pyspark.sql.functions import (
    col, trim, lower, length, lit, explode,
    split, regexp_replace
)

# 1. Carrega os créditos da Bronze
df_creditos = spark.table("workspace.bronze.tb_credits_and_tags")


# 2. Identifica os registros com deslocamento de campos já investigados
df_creditos_suspeitos = df_creditos.filter(
    (
        (length(trim(col("genres"))) > 100) &
        (length(trim(col("production_companies"))) > 100)
    )
    |
    col("id").isin("522631", "461388", "1326649")
)

ids_suspeitos = df_creditos_suspeitos.select("id").distinct()

print("Registros suspeitos identificados:", df_creditos_suspeitos.count())
print("Filmes distintos afetados:", ids_suspeitos.count())


# 3. Exclui esses IDs somente da extração de pessoas e empresas
# A tabela Bronze permanece inalterada
df_creditos_validos = df_creditos.join(
    ids_suspeitos,
    on="id",
    how="left_anti"
)

print("Registros disponíveis para extração:", df_creditos_validos.count())


# 4. Função de extração e normalização
def extrair_entidades(df, coluna, tipo):
    return (
        df
        .select(
            trim(col("id")).alias("id"),

            explode(
                split(
                    regexp_replace(
                        col(coluna),
                        r"\s*[|;]\s*",
                        ","
                    ),
                    ","
                )
            ).alias("nome"),

            col("ingestion_datetime")
        )

        .withColumn("nome", trim(col("nome")))
        .withColumn("tipo", lit(tipo))

        # Remove campos vazios e marcadores de ausência
        .filter(
            col("nome").isNotNull() &
            (col("nome") != "") &
            (~lower(col("nome")).isin(
                "n/a", "na", "null", "none", "unknown", "-"
            ))
        )

        # Descarta fragmentos muito extensos
        .filter(length(col("nome")) <= 100)

        # Descarta fragmentos com sinais de texto quebrado do CSV
        .filter(
            ~col("nome").rlike(r'[\\\r\n]')
        )
    )


# 5. Extrai empresas, diretores, roteiristas e atores
df_empresas = extrair_entidades(
    df_creditos_validos,
    "production_companies",
    "empresa"
)

df_diretores = extrair_entidades(
    df_creditos_validos,
    "directors",
    "diretor"
)

df_roteiristas = extrair_entidades(
    df_creditos_validos,
    "writers",
    "roteirista"
)

df_elenco = extrair_entidades(
    df_creditos_validos,
    "cast",
    "ator"
)


# 6. Reúne as entidades em uma única tabela Silver
df_pessoas_empresas_silver = (
    df_empresas
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_elenco)
    .dropDuplicates(["id", "nome", "tipo"])
)


# 7. Confere o resultado
print(
    "Registros na tabela Silver de pessoas e empresas:",
    df_pessoas_empresas_silver.count()
)

display(df_pessoas_empresas_silver.limit(30))

In [0]:
# Carrega e verifica a cotação do dólar da camada Bronze
df_cotacao = spark.table("workspace.bronze.tb_cotacao_dolar")

df_cotacao.printSchema()

display(df_cotacao)



In [0]:
# Converte a data/hora da cotação para timestamp e cria a coluna de data que será usada para ordenar e tratar as cotações na camada Silver.

from pyspark.sql.functions import to_timestamp, to_date, col

df_cotacao_tratada = (
    df_cotacao
    .withColumn(
        "dataHoraCotacao",
        to_timestamp(col("dataHoraCotacao"), "yyyy-MM-dd HH:mm:ss.SSSSSS")
    )
    .withColumn(
        "data_cotacao",
        to_date(col("dataHoraCotacao"))
    )
)

display(df_cotacao_tratada.orderBy("data_cotacao"))



In [0]:
# Cria o calendário e preenche os dias sem nova cotação

from pyspark.sql.functions import (
    col, sequence, explode, min, max, last, row_number,
    desc_nulls_last
)
from pyspark.sql.window import Window

# Mantém a cotação mais recente de cada dia

janela_diaria = Window.partitionBy("data_cotacao").orderBy(
    desc_nulls_last("dataHoraCotacao"),
    desc_nulls_last("ingestion_datetime"),
    desc_nulls_last("cotacaoVenda"),
    desc_nulls_last("cotacaoCompra")
)

df_cotacao_diaria = (
    df_cotacao_tratada
    .filter(col("data_cotacao").isNotNull())
    .withColumn("rn", row_number().over(janela_diaria))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Cria uma linha para cada dia do período disponível

df_intervalo = df_cotacao_diaria.select(
    min("data_cotacao").alias("data_inicio"),
    max("data_cotacao").alias("data_fim")
)

df_calendario = df_intervalo.select(
    explode(
        sequence(col("data_inicio"), col("data_fim"))
    ).alias("data_cotacao")
)

df_cotacao_completa = df_calendario.join(
    df_cotacao_diaria,
    on="data_cotacao",
    how="left"
)

# Usa a última cotação conhecida nos dias sem atualização

janela_cotacao = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_cotacao_silver = (
    df_cotacao_completa
    .withColumn(
        "cotacaoCompra",
        last("cotacaoCompra", ignorenulls=True).over(janela_cotacao)
    )
    .withColumn(
        "cotacaoVenda",
        last("cotacaoVenda", ignorenulls=True).over(janela_cotacao)
    )
    .withColumn(
        "dataHoraCotacao",
        last("dataHoraCotacao", ignorenulls=True).over(janela_cotacao)
    )
    .withColumn(
        "ingestion_datetime",
        last("ingestion_datetime", ignorenulls=True).over(janela_cotacao)
    )
)


In [0]:
# Organiza a estrutura final da cotação para a camada Silver.

df_cotacao_silver = (
    df_cotacao_silver
    .select(
        col("data_cotacao"),
        col("cotacaoCompra").alias("cotacao_compra"),
        col("cotacaoVenda").alias("cotacao_venda"),
        col("dataHoraCotacao").alias("data_hora_cotacao"),
        col("ingestion_datetime")
    )
    .orderBy("data_cotacao")
)

df_cotacao_silver.printSchema()
display(df_cotacao_silver)



In [0]:
# Grava os DataFrames tratados como tabelas Delta na camada Silver.

df_info_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.tb_info_filmes")

df_financeiro_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.tb_financeiro_filmes")

df_metricas_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.tb_metricas_engajamento")

df_avaliacoes_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.tb_avaliacoes_usuarios")

df_generos_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.tb_generos")

df_pessoas_empresas_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.tb_pessoas_empresas")

df_cotacao_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.tb_cotacao_dolar")

print("As 7 tabelas Silver foram gravadas com sucesso.")

